# LAB | Extractive Question Answering

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

In [1]:
!pip uninstall torchcodec -y -q

In [2]:
!pip install torchvision==0.26.0+cu130 \
    --index-url https://download.pytorch.org/whl/cu130 \
    --force-reinstall -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 116.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 149.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 141.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 207.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 103.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 251.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 129.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 244.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 256.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 203.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 123.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 167.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch, torchvision
print(f"PyTorch:     {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
# Both should end in +cu130 ✅

PyTorch:     2.11.0+cu130
torchvision: 0.26.0+cu130
CUDA:        True


# Install Dependencies

In [4]:
# uninstall torchcodec before any imports
!pip uninstall torchcodec -y -q

In [5]:
# now safe to import
!pip install -qU datasets pinecone-client sentence-transformers transformers
!pip install -qU langchain-pinecone pinecone-notebooks

In [6]:
import os
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [17]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("squad", split="train").to_pandas()

In [18]:
# select only title and context column
df = df[["title", "context"]]
# drop rows containing duplicate context passages
df = df.drop_duplicates(subset="context").reset_index(drop=True)


# Initialize Pinecone Index

The Pinecone index stores vector representations of our context passages which we can retrieve using another vector (query vector). We first need to initialize our connection to Pinecone to create our vector index. For this, we need a free [API key]("https://app.pinecone.io/"), and then we initialize the connection like so:

In [27]:
from pinecone import Pinecone, ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# # connect to pinecone environment
pc = Pinecone(
     api_key = PINECONE_API_KEY,
     environment='us-east-1'  # find next to API key in console
 )



Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [50]:
index_name = "question-answering"

# check if the extractive-question-answering index exists
if index_name not in pc.list_indexes().names():
    # create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec = spec
    )
# connect to extractive-question-answering index we created
index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [60]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1") #use the 'multi-qa-MiniLM-L6-cos-v1' model from HuggingFace to build the retriever
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [61]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(df))

    # extract batch
    batch = df.iloc[i:i_end]

    # generate embeddings for batch
    emb = retriever.encode(batch["context"].tolist()).tolist()

    # get metadata
    meta = batch.to_dict(orient="records")

    # create unique IDs
    ids = [str(x) for x in range(i, i_end)]

    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))

    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891,
 'vector_type': 'dense'}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [62]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = 'deepset/electra-base-squad2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)

def reader(question, context):
    inputs = tokenizer(question, context, return_tensors="pt",
                      truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = outputs.start_logits.argmax()
    end = outputs.end_logits.argmax() + 1
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )
    score = float(outputs.start_logits.max() + outputs.end_logits.max())
    return {"answer": answer, "score": score}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [63]:
from pprint import pprint

def get_context(question, top_k):
    xq = retriever.encode([question]).tolist()
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return [match["metadata"]["context"] for match in xc["matches"]]


In [64]:
from pprint import pprint

def extract_answer(question, context):
    results = []
    for c in context:
        answer = reader(question=question, context=c)
        answer["context"] = c
        results.append(answer)
    sorted_result = sorted(results, key=lambda x: x['score'], reverse=True)
    print(sorted_result)
    return sorted_result

In [65]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

In [43]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [66]:
extract_answer(question, context)

[{'answer': '691, 000 bbl / d', 'score': 21.08501434326172, 'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]


[{'answer': '691, 000 bbl / d',
  'score': 21.08501434326172,
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [45]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'hurley and chen', 'score': 19.325923919677734, 'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]


[{'answer': 'hurley and chen',
  'score': 19.325923919677734,
  'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]

In [46]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity', 'score': 12.574865341186523, 'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]


[{'answer': 'his theories of special relativity and general relativity',
  'score': 12.574865341186523,
  'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]

Let's run another question. This time for top 3 context passages from the retriever.

In [47]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'armstrong', 'score': 11.328462600708008, 'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'}, {'answer': 'frank borman', 'score': 4.50399923324585, 'context': "On December 21, 1968, Frank Borman, James Lovell, and William Anders became the first humans to r

[{'answer': 'armstrong',
  'score': 11.328462600708008,
  'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'},
 {'answer': 'frank borman',
  'score': 4.50399923324585,
  'context': "On December 21, 1968, Frank Borman, James Lovell, and William Anders became the first hu

The result looks pretty good.

In [48]:
pc.delete_index(index_name)

### Add a few more questions. What did you observe?

In [67]:
question = "Who started facebook?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': '[CLS] who started facebook? [SEP]', 'score': 14.189208030700684, 'context': 'The first wave of modern Jewish migration to Ottoman-ruled Palestine, known as the First Aliyah, began in 1881, as Jews fled pogroms in Eastern Europe. Although the Zionist movement already existed in practice, Austro-Hungarian journalist Theodor Herzl is credited with founding political Zionism, a movement which sought to establish a Jewish state in the Land of Israel, thus offering a solution to the so-called Jewish Question of the European states, in conformity with the goals and achievements of other national projects of the time. In 1896, Herzl published Der Judenstaat (The State of the Jews), offering his vision of a future Jewish state; the following year he presided over the first Zionist Congress.'}, {'answer': '', 'score': 14.132329940795898, 'context': "The first web browser was invented in 1990 by Sir Tim Berners-Lee. Berners-Lee is the director of the World Wide Web Consortium (W3C), 

[{'answer': '[CLS] who started facebook? [SEP]',
  'score': 14.189208030700684,
  'context': 'The first wave of modern Jewish migration to Ottoman-ruled Palestine, known as the First Aliyah, began in 1881, as Jews fled pogroms in Eastern Europe. Although the Zionist movement already existed in practice, Austro-Hungarian journalist Theodor Herzl is credited with founding political Zionism, a movement which sought to establish a Jewish state in the Land of Israel, thus offering a solution to the so-called Jewish Question of the European states, in conformity with the goals and achievements of other national projects of the time. In 1896, Herzl published Der Judenstaat (The State of the Jews), offering his vision of a future Jewish state; the following year he presided over the first Zionist Congress.'},
 {'answer': '',
  'score': 14.132329940795898,
  'context': "The first web browser was invented in 1990 by Sir Tim Berners-Lee. Berners-Lee is the director of the World Wide Web Consortiu

In [68]:
question = "Who is the CEO of microsoft?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': '[CLS]', 'score': 14.35166072845459, 'context': 'Politically, the paper\'s stance was less clear under Prime Minister Gordon Brown who succeeded Blair in June 2007. Its editorials were critical of many of Brown\'s policies and often more supportive of those of Conservative leader David Cameron. Rupert Murdoch, head of The Sun\'s parent company News Corporation, speaking at a 2007 meeting with the House of Lords Select Committee on Communications, which was investigating media ownership and the news, said that he acts as a "traditional proprietor". This means he exercises editorial control on major issues such as which political party to back in a general election or which policy to adopt on Europe.'}, {'answer': '[CLS]', 'score': 14.225057601928711, 'context': 'On October 11, 2011, Doug Morris announced that Mel Lewinter had been named Executive Vice President of Label Strategy. Lewinter previously served as chairman and CEO of Universal Motown Republic Group. In January 20

[{'answer': '[CLS]',
  'score': 14.35166072845459,
  'context': 'Politically, the paper\'s stance was less clear under Prime Minister Gordon Brown who succeeded Blair in June 2007. Its editorials were critical of many of Brown\'s policies and often more supportive of those of Conservative leader David Cameron. Rupert Murdoch, head of The Sun\'s parent company News Corporation, speaking at a 2007 meeting with the House of Lords Select Committee on Communications, which was investigating media ownership and the news, said that he acts as a "traditional proprietor". This means he exercises editorial control on major issues such as which political party to back in a general election or which policy to adopt on Europe.'},
 {'answer': '[CLS]',
  'score': 14.225057601928711,
  'context': 'On October 11, 2011, Doug Morris announced that Mel Lewinter had been named Executive Vice President of Label Strategy. Lewinter previously served as chairman and CEO of Universal Motown Republic Group. In J

Observation:
- bad retrieved context
- answer extractor forced to answer